# Estrazione delle fonti dal dataset

This cell loads the dataset [Croc-Prog-HF/Creative-knowledge-for-Writing](https://huggingface.co/datasets/Croc-Prog-HF/Creative-knowledge-for-Writing) from Hugging Face and reconstructs the relationship between Source ID and Source Label. This is necessary because the dataset defines the Source column as ClassLabel, so only the label's numeric index is saved in the data.

The result will look something like this:
| Source_ID | Source_Name |
|---|---|
|0|Hunger Games|
|1|The Philosopher's of Life|
|2|The Red Pyramid|

After processing it will be exported as a CSV file.

## Installazione delle librerie
```python
!pip install -q datasets pandas duckdb
```

This notebook was created to map and index all the sources used in this dataset, providing maximum transparency into the sources used and possibly filtering out unwanted sources.

In [ ]:
# Install libraries (run safely even if already installed)
!pip install -q datasets pandas duckdb

from datasets import load_dataset
import pandas as pd
import duckdb

# Load dataset from HuggingFace
dataset = load_dataset("Croc-Prog-HF/Creative-knowledge-for-Writing")

# Extract Source label names (ClassLabel metadata)
source_names = dataset["train"].features["Source"].names

# Build Source_ID -> Source_Name table
source_labels = pd.DataFrame({
    "Source_ID": range(len(source_names)),
    "Source_Name": source_names
})

# Convert dataset split to pandas
train_df = dataset["train"].to_pandas()

# Create DuckDB connection
con = duckdb.connect()

# Register tables inside DuckDB
con.register("train", train_df)
con.register("source_labels", source_labels)

# SQL query to get unique sources used in the dataset
result = con.execute("""
SELECT DISTINCT
    t.Source AS Source_ID,
    s.Source_Name
FROM train t
JOIN source_labels s
ON t.Source = s.Source_ID
ORDER BY Source_ID
""").df()

# Show result
print("Unique Sources Used:")
display(result)

# Save mapping to CSV
source_labels.to_csv("source_labels.csv", index=False)

print(f"\nTotal sources defined in dataset: {len(source_labels)}")
print(f"Sources actually used in train split: {len(result)}")

d:\Python\Python3.11\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
d:\Python\Python3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Python\Python3.11\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\princ\.cache\huggingface\hub\datasets--Croc-Prog-HF--Creative-knowledge-for-Writing. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see ht

Unique Sources Used:


,Source_ID,Source_Name
0,0,1-Hunger games
1,1,1-The Philosopher's Stone
2,2,1-The Red Pyramid.pdf
3,3,1001 Afternoons in Chicago
4,4,2-Catching Fire.pdf
...,...,...
186,186,Vulcan's Workshop
187,187,Watch the Sky
188,188,Werewolves of War
189,189,When Sea Becomes Sky



Total sources defined in dataset: 191
Sources actually used in train split: 191
